# Ej. 4 · Mejoras (1,5 puntos)

Dos de las ideas del enunciado, baratas de medir y con explicación clara:

- **Idea B:** no dejar volumen pendiente y consumir ≈90 % de la duración, con calibración de las recompensas (D-4.1).
- **Idea C:** parar el entrenamiento según el aprendizaje, guardar los pesos cada cierto tiempo y poder recuperarlo (D-4.2).
- **Idea A** (estado con libro y tendencia, recompensa local de precio) queda preparada en `rewards.TrackingPrice` y el estado
  `TVd`, si hay tiempo.

### Qué cambia respecto al ej. 3

| Bloque | Cambio | Decisión |
|---|---|---|
| Recompensa | `Tracking(horizon=0.9, pending_penalty=k)`: el plan acaba al 90 % y lo pendiente se penaliza al final | D-4.1 |
| Bucle | Checkpoints (`*_best.pt`, `*_last.pt`), parada por objetivo o por estancamiento, reanudación | D-4.2 |

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))   # ejecutar desde notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import data as dt, rewards as rw, exploration as ex, protocol as pr, plots as pl
from common.env import BestExecutionEnv
from common.agents import DDQNAgent, TWAPAgent
pd.set_option("display.precision", 3)
%matplotlib inline

In [ ]:
ASSET = "REP"                      # D-0.2
DATA = dt.load(ASSET)
train = DATA["train"]
GRID = pr.make_grid(train, n_days=3, seed=0)   # rejilla fija de evaluación (D-0.3)
print(dt.summary(DATA)); print(len(GRID), "episodios de evaluación")

## Idea B · Sin volumen pendiente y ≈90 % de la duración

El plan objetivo pasa a ser un TWAP que acaba en 0,9 · N: $\mathrm{plan}_t = \min\big(n,\ t\,n / (0{,}9\,N)\big)$. Así el
algoritmo deja un 10 % de margen. Un término final $-k \cdot \text{pendiente}/n$ castiga terminar con la orden a medias.
Se calibra k: si es muy grande, el agente acelera y acaba demasiado pronto (aviso de clase: «termina antes, pero demasiado antes»).

In [ ]:
CFG = dict(gamma=0.9, alpha=1e-3, buffer_size=50_000, batch_size=64, hidden_neurons=32)
twap_err = pr.load_df("00_twap_grid").max_err.mean()

def run(reward, seed=0, nep=300, **train_kw):
    pr.set_seeds(seed)
    env = BestExecutionEnv(train, reward=reward, seed=seed)
    agent = DDQNAgent(env, seed=seed, **CFG)
    pr.fill_buffer(agent, env, 25)
    ep, ev = pr.train(agent, env, nep, ex.ImprovementEpsilon(target_error=twap_err), 8, 1000, eval_data=train,
                      eval_grid=GRID, eval_every=25, eval_reward=reward, log_every=0, verbose=False, **train_kw)
    return agent, ep, ev, pr.evaluate(agent, train, GRID, reward=reward)

RUN = not (pr.RESULTS / "ej4_ideaB.csv").exists()
if RUN:
    rows = []
    for k in (0.0, 1.0, 5.0):
        for seed in range(2):
            _, _, _, res = run(rw.Tracking(horizon=0.9, pending_penalty=k), seed)
            rows.append(res.assign(k=k, seed=seed))
            print(k, seed, res.t_fin.mean().round(3), res.pending.mean())
    pr.save_df(pd.concat(rows), "ej4_ideaB")
b = pr.load_df("ej4_ideaB")
b.groupby("k")[["t_fin", "pending", "completed", "max_err"]].agg(["mean", "std"])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for k, g in b.groupby("k"):
    ax.hist(g.t_fin, bins=20, alpha=0.5, label=f"k = {k}")
ax.axvline(0.9, c="k", ls="--", lw=1); ax.set_xlabel("instante de finalización (t / N)"); ax.legend()
ax.set_title("Idea B · cuándo termina la orden")
pl.save(fig, "ej4_ideaB_tfin")

## Idea C · Parada por aprendizaje, checkpoints y recuperación

- **Cuándo parar:** si la métrica greedy (error máximo medio) baja de 1,5 títulos, el objetivo está cumplido; si no mejora
  0,05 títulos en 4 evaluaciones seguidas, el aprendizaje se ha estancado.
- **Checkpoints:** en cada evaluación se guarda el último modelo y, si mejora, el mejor. Protege del *catastrophic forgetting*.
- **Recuperar:** se carga el checkpoint y se continúa con `start_episode`.

In [ ]:
ckpt = pr.MODELS / "ej4_ideaC"
agent, ep, ev, res = run(rw.Tracking(), seed=0, nep=600, checkpoint=str(ckpt), stop_target=1.5, stop_patience=4)
print(f"Paró en el episodio {ep.episode.max() + 1}; mejor métrica {ev.max_err.min():.2f}")
pr.save_df(ev, "ej4_ideaC_evals")
pl.plot_training(ep, ev, title="Idea C · entrenamiento con parada", save="ej4_ideaC_training");

In [ ]:
# Recuperación: un agente nuevo carga el mejor checkpoint y continúa 50 episodios más
env = BestExecutionEnv(train, reward=rw.Tracking(), seed=1)
resumed = DDQNAgent(env, seed=1, **CFG)
info = resumed.load(f"{ckpt}_best.pt")
print("checkpoint del episodio", info["episode"], "· métrica", round(info["metric"], 2))
before = pr.evaluate(resumed, train, GRID, reward=rw.Tracking()).max_err.mean()
pr.fill_buffer(resumed, env, 10)
ep2, ev2 = pr.train(resumed, env, 50, ex.LinearDecay(resumed.epsilon, 0.05, epsilon_decay=0.0), 8, 1000, eval_data=train,
                    eval_grid=GRID, eval_every=25, eval_reward=rw.Tracking(), start_episode=info["episode"], verbose=False)
print(f"error máx. al recuperar: {before:.2f} → tras continuar: {ev2.max_err.iloc[-1]:.2f}")

## Conclusión

Completar con las cifras: k elegido y por qué (instante de finalización ≈ 0,9 sin pendiente), episodio de parada y
comprobación de que el modelo recuperado conserva la métrica.